# Unit 7 Hands-On: MA-POCA — SoccerTwos (AI vs AI)

2vs2 축구 팀을 **MA-POCA(Multi-Agent POsthumous Credit Assignment)** 알고리즘과  
**Self-Play**로 훈련하여 상대 팀을 이기는 에이전트를 만듭니다.

훈련된 모델은 HF Hub에 업로드되어 **AI vs AI 리더보드**에 자동 등록됩니다.

### Unit 5(ML-Agents)와의 차이점

| 항목 | Unit 5 | Unit 7 |
|---|---|---|
| **에이전트 수** | 1개 | **4개 (2팀 × 2명)** |
| **알고리즘** | PPO | **MA-POCA + Self-Play** |
| **훈련 시간** | 10~35분 | **4~8시간** |
| **평가 방식** | 보상 점수 | **ELO 레이팅 (AI vs AI)** |

### MA-POCA란?
```
기존 PPO: 각 에이전트가 독립적으로 학습 → 팀 협동 어려움
MA-POCA: 중앙 Critic이 팀 전체 상태를 보고 각 에이전트 기여도를 평가
         → 각 에이전트는 로컬 관측으로만 행동 (분산 실행)
         → 팀 협동 행동 자연스럽게 학습
```

> ⚠️ 강의에서는 **로컬 PC 훈련을 권장**합니다 (4~8시간 소요).  
> 이 노트북은 Colab + Miniconda 환경에서도 실행 가능하도록 구성했습니다.  
> Colab 세션 제한(12시간)을 고려해 `max_steps`를 조정하세요.

---
## 목차
1. Miniconda 설치 및 ML-Agents 환경 구성
2. Google Drive 마운트
3. ML-Agents 설치
4. SoccerTwos 환경 다운로드
5. 환경 이해
6. MA-POCA 설정 파일(YAML) 생성
7. 에이전트 훈련
8. 훈련 결과 확인 (TensorBoard)
9. Hugging Face Hub 업로드
10. AI vs AI 챌린지 참가 확인


---
## 1. Miniconda 설치 및 ML-Agents 환경 구성

ML-Agents는 Python 3.10.12가 필요합니다.  
Unit 5와 동일하게 Miniconda로 격리된 환경을 구성합니다.


In [1]:
import os
import sys
import shutil
import subprocess
import urllib.request

MINICONDA_DIR = '/content/miniconda3'
ENV_NAME      = 'mlagents'

if 'google.colab' in sys.modules:
    candidate_paths = [
        os.path.join(MINICONDA_DIR, 'bin', 'conda'),
        os.path.expanduser('~/miniconda3/bin/conda'),
        '/usr/local/bin/conda',
    ]
    conda_path = next((p for p in candidate_paths if os.path.exists(p)), None)

    if not conda_path:
        print('Colab 환경에서 Miniconda를 설치합니다...')
        installer = '/tmp/miniconda.sh'
        urls = [
            'https://repo.anaconda.com/miniconda/Miniconda3-py310_24.11.3-0-Linux-x86_64.sh',
            'https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh',
        ]
        downloaded = False
        for url in urls:
            try:
                print(f'다운로드 시도: {url}')
                urllib.request.urlretrieve(url, installer)
                downloaded = True
                print('✅ 다운로드 완료')
                break
            except Exception as e:
                print(f'⚠️ 실패: {url} -> {e}')

        if not downloaded:
            raise RuntimeError('Miniconda 설치 파일을 다운로드하지 못했습니다.')

        result = subprocess.run(
            ['bash', installer, '-b', '-p', MINICONDA_DIR],
            capture_output=True, text=True, check=False,
        )
        if result.stdout: print(result.stdout)
        if result.stderr: print(result.stderr)
        conda_path = next((p for p in candidate_paths if os.path.exists(p)), None)

    if not conda_path:
        raise RuntimeError('Miniconda 설치가 완료되지 않았습니다.')

    print('conda 경로:', conda_path)

    # conda 채널 약관 승인
    for channel in [
        'https://repo.anaconda.com/pkgs/main',
        'https://repo.anaconda.com/pkgs/r',
    ]:
        subprocess.run(
            [conda_path, 'tos', 'accept', '--override-channels', '--channel', channel],
            capture_output=True, text=True, check=False,
        )

    # Python 3.10.12 환경 생성
    try:
        subprocess.run(
            [conda_path, 'create', '-y', '-n', ENV_NAME, 'python=3.10.12', 'ujson'],
            capture_output=True, text=True, check=True,
        )
        print(f'✅ conda 환경 생성 완료: {ENV_NAME}')
    except subprocess.CalledProcessError as e:
        if 'already exists' in (e.stdout or '') + (e.stderr or ''):
            print(f'ℹ️  이미 존재하는 환경입니다: {ENV_NAME}')
        else:
            print('❌ conda 환경 생성 실패')
            print(e.stdout); print(e.stderr)
            raise

else:
    print('Colab 환경이 아닙니다.')
    if shutil.which('conda'):
        print(f'로컬 conda로 환경을 생성하세요:')
        print(f'  conda create -y -n {ENV_NAME} python=3.10.12 ujson')
        print(f'  conda activate {ENV_NAME}')
    else:
        print('Miniconda를 먼저 설치하세요: https://www.anaconda.com/download/success')


Colab 환경에서 Miniconda를 설치합니다...
다운로드 시도: https://repo.anaconda.com/miniconda/Miniconda3-py310_24.11.3-0-Linux-x86_64.sh
⚠️ 실패: https://repo.anaconda.com/miniconda/Miniconda3-py310_24.11.3-0-Linux-x86_64.sh -> HTTP Error 404: Not Found
다운로드 시도: https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
✅ 다운로드 완료
PREFIX=/content/miniconda3
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /content/miniconda3

conda 경로: /content/miniconda3/bin/conda
✅ conda 환경 생성 완료: mlagents


In [2]:
import os, subprocess

MINICONDA_DIR = '/content/miniconda3'
ENV_NAME      = 'mlagents'
conda_path    = os.path.join(MINICONDA_DIR, 'bin', 'conda')
conda_python  = os.path.join(MINICONDA_DIR, 'envs', ENV_NAME, 'bin', 'python')
conda_pip     = os.path.join(MINICONDA_DIR, 'envs', ENV_NAME, 'bin', 'pip')

if os.path.exists(conda_python):
    result = subprocess.run([conda_python, '--version'], capture_output=True, text=True)
    print('conda Python:', result.stdout.strip())
else:
    print(f'❌ conda Python이 없습니다: {conda_python}')
    print('이전 셀을 다시 실행하세요.')

# 이후 셀에서 참조할 수 있도록 환경변수로 저장
os.environ['CONDA_PYTHON'] = conda_python
os.environ['CONDA_PIP']    = conda_pip
os.environ['ENV_NAME']     = ENV_NAME
print(f'CONDA_PYTHON: {conda_python}')
print(f'CONDA_PIP:    {conda_pip}')


conda Python: Python 3.10.12
CONDA_PYTHON: /content/miniconda3/envs/mlagents/bin/python
CONDA_PIP:    /content/miniconda3/envs/mlagents/bin/pip


---
## 2. Google Drive 마운트

훈련 결과는 ML-Agents 기본 경로(`/content/ml-agents/results`)에 저장됩니다.  
훈련 완료 후 Drive로 복사하여 세션 종료 후에도 보존합니다.

```
훈련 중 저장 경로: /content/ml-agents/results/SoccerTwos/
훈련 완료 후 백업: Google Drive/RL_Course/Unit7_SoccerTwos/results/SoccerTwos/
```


In [3]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ✏️ Drive 저장 경로를 원하는 대로 변경하세요.
DRIVE_BASE    = '/content/drive/MyDrive/RL_Course/Unit7_SoccerTwos'
DRIVE_RESULTS = f'{DRIVE_BASE}/results'

os.makedirs(DRIVE_RESULTS, exist_ok=True)

# 훈련 결과는 ML-Agents 기본 경로(/content/ml-agents/results)에 저장됩니다.
# 훈련 완료 후 Drive로 복사하는 방식을 사용합니다. (심볼릭 링크 사용 안 함)
LOCAL_RESULTS = '/content/ml-agents/results'

os.environ['DRIVE_BASE']    = DRIVE_BASE
os.environ['DRIVE_RESULTS'] = DRIVE_RESULTS
os.environ['LOCAL_RESULTS'] = LOCAL_RESULTS

print('✅ Drive 마운트 완료!')
print(f'   훈련 결과 (로컬) : {LOCAL_RESULTS}')
print(f'   Drive 백업 경로  : {DRIVE_RESULTS}')


Mounted at /content/drive
✅ Drive 마운트 완료!
   훈련 결과 (로컬) : /content/ml-agents/results
   Drive 백업 경로  : /content/drive/MyDrive/RL_Course/Unit7_SoccerTwos/results


---
## 3. ML-Agents 설치

ML-Agents 소스코드를 클론한 뒤 conda 환경의 pip으로 설치합니다.  
conda 환경의 pip을 직접 지정하여 Python 3.10에 설치되도록 합니다.


In [4]:
import subprocess, os

# ML-Agents 소스코드 클론 (약 2~3분, 2.63GB)
if not os.path.exists('/content/ml-agents'):
    print('ML-Agents 소스코드 클론 중... (약 2~3분 소요)')
    result = subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/Unity-Technologies/ml-agents',
         '/content/ml-agents'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print('✅ 클론 완료')
    else:
        print('❌ 클론 실패:', result.stderr)
else:
    print('ℹ️  이미 클론되어 있습니다.')


ML-Agents 소스코드 클론 중... (약 2~3분 소요)
✅ 클론 완료


In [5]:
import subprocess, os

conda_pip = os.environ.get('CONDA_PIP',
    '/content/miniconda3/envs/mlagents/bin/pip')

# conda 환경의 pip으로 ML-Agents 설치
# (Colab 기본 Python이 아닌 conda Python 3.10에 설치)
for pkg_path in [
    '/content/ml-agents/ml-agents-envs',
    '/content/ml-agents/ml-agents',
]:
    print(f'설치 중: {pkg_path}')
    result = subprocess.run(
        [conda_pip, 'install', '-e', pkg_path],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f'  ✅ 완료')
    else:
        print(f'  ❌ 실패')
        print(result.stderr[-500:])


설치 중: /content/ml-agents/ml-agents-envs
  ✅ 완료
설치 중: /content/ml-agents/ml-agents
  ✅ 완료


In [6]:
import subprocess, os

conda_python = os.environ.get('CONDA_PYTHON',
    '/content/miniconda3/envs/mlagents/bin/python')

# mlagents-learn 명령어가 정상 설치됐는지 확인
mlagents_bin = os.path.join(
    os.path.dirname(conda_python), 'mlagents-learn'
)

if os.path.exists(mlagents_bin):
    result = subprocess.run(
        [mlagents_bin, '--help'],
        capture_output=True, text=True
    )
    # 버전 정보 첫 줄만 출력
    first_line = result.stdout.strip().split('\n')[0] if result.stdout else ''
    print('✅ mlagents-learn 설치 확인:', first_line)
else:
    print(f'❌ mlagents-learn을 찾을 수 없습니다: {mlagents_bin}')
    print('이전 셀(ML-Agents 설치)을 다시 실행하세요.')

# 이후 셀에서 사용할 경로 환경변수 저장
os.environ['MLAGENTS_LEARN'] = mlagents_bin
os.environ['MLAGENTS_PUSH']  = os.path.join(
    os.path.dirname(conda_python), 'mlagents-push-to-hf'
)
print(f'MLAGENTS_LEARN: {mlagents_bin}')


✅ mlagents-learn 설치 확인: usage: mlagents-learn [-h] [--env ENV_PATH] [--resume] [--deterministic]
MLAGENTS_LEARN: /content/miniconda3/envs/mlagents/bin/mlagents-learn


---
## 4. SoccerTwos 환경 다운로드

SoccerTwos 실행 파일을 운영체제에 맞게 다운로드합니다.  
Colab은 Linux(Ubuntu) 환경이므로 Linux 실행 파일을 사용합니다.

| OS | 다운로드 링크 |
|---|---|
| **Linux (Colab)** | 아래 셀에서 자동 다운로드 |
| Windows | [Google Drive](https://drive.google.com/file/d/1sqFxbEdTMubjVktnV4C6ICjp89wLhUcP/view) |
| Mac | [Google Drive](https://drive.google.com/drive/folders/1h7YB0qwjoxxghApQdEUQmk95ZwIDxrPG) |


In [11]:
import subprocess, os, glob

EXEC_DIR = '/content/ml-agents/training-envs-executables/linux'
ZIP_PATH = f'{EXEC_DIR}/SoccerTwos.zip'
ENV_DIR  = f'{EXEC_DIR}/SoccerTwos'

os.makedirs(EXEC_DIR, exist_ok=True)

# Linux용 SoccerTwos 환경 다운로드
if not os.path.exists(ENV_DIR):
    print('SoccerTwos 환경 다운로드 중...')

    # gdown으로 Google Drive에서 다운로드
    subprocess.run(['pip', 'install', 'gdown', '-q'], check=False)
    result = subprocess.run([
        'gdown',
        'https://drive.google.com/uc?id=1KuqBKYiXiIcU4kNMqEzhgypuFP5_45CL',
        '-O', ZIP_PATH,
    ], capture_output=True, text=True)

    if result.returncode != 0:
        print('❌ 다운로드 실패:', result.stderr[-200:])
    else:
        print('✅ 다운로드 완료, 압축 해제 중...')
        subprocess.run(['unzip', '-q', '-d', EXEC_DIR, ZIP_PATH], check=False)
        subprocess.run(['chmod', '-R', '755', EXEC_DIR], check=False)
        print('✅ 압축 해제 완료')
else:
    print('ℹ️  SoccerTwos 환경이 이미 존재합니다.')

# 실제 실행 파일 자동 탐색 (.x86_64 또는 확장자 없는 바이너리)
candidates = (
    glob.glob(f'{EXEC_DIR}/**/*.x86_64', recursive=True) +
    glob.glob(f'{EXEC_DIR}/**/SoccerTwos',  recursive=True)
)
# 디렉토리 제외, 실행 가능한 파일만
exec_files = [p for p in candidates if os.path.isfile(p) and os.access(p, os.X_OK)]

if exec_files:
    SOCCER_ENV = exec_files[0]
    os.environ['SOCCER_ENV'] = SOCCER_ENV
    print(f'\n✅ 실행 파일 발견: {SOCCER_ENV}')
else:
    # 실행 파일 목록 출력 (디버깅용)
    print('\n❌ 실행 파일을 찾지 못했습니다. 압축 해제된 파일 목록:')
    for root, dirs, files in os.walk(EXEC_DIR):
        for f in files:
            print(f'  {os.path.join(root, f)}')

SoccerTwos 환경 다운로드 중...
✅ 다운로드 완료, 압축 해제 중...
✅ 압축 해제 완료

✅ 실행 파일 발견: /content/ml-agents/training-envs-executables/linux/SoccerTwos.x86_64


---
## 5. 환경 이해

### SoccerTwos
2팀 × 2명 = 4개 에이전트가 2vs2 축구를 합니다.

**관측 공간** (336차원 벡터):
- 전방 11개 레이캐스트 (120도 범위) → 264차원
- 후방 3개 레이캐스트 (90도 범위) → 72차원
- 각 레이캐스트가 감지하는 객체: 공, 파란 골대, 보라 골대, 벽, 파란 에이전트, 보라 에이전트

**행동 공간** (3개 이산 브랜치):

| 브랜치 | 행동 |
|---|---|
| 앞/뒤 이동 | 정지, 앞으로, 뒤로 |
| 회전 | 정지, 시계방향, 반시계방향 |
| 좌/우 이동 | 정지, 오른쪽, 왼쪽 |

**보상 구조**:
- 골 성공: **+1 - 페널티** (페널티는 시간에 따라 증가)
- 골 실패: **-1**
- 실수 (잘못된 방향 이동 등): 작은 패널티

### Self-Play + MA-POCA
```
Self-Play: 에이전트가 과거 자신(체크포인트)과 대전하며 점진적으로 강해짐
MA-POCA:   팀원 간 협동 행동을 중앙 Critic이 평가 → 개인 기여도 분리

결과: 팀 협동 + 상대 팀 대응 능력을 동시에 학습
```


---
## 6. MA-POCA 설정 파일(YAML) 생성

| 파라미터 | 값 | 설명 |
|---|---|---|
| `trainer_type` | poca | MA-POCA 알고리즘 |
| `max_steps` | 5,000,000 | 총 훈련 스텝 (권장: 5M~10M) |
| `batch_size` | 2048 | 미니배치 크기 |
| `buffer_size` | 20480 | 경험 버퍼 크기 |
| `hidden_units` | 512 | 신경망 은닉층 크기 |
| `time_horizon` | 1000 | 한 번에 수집하는 스텝 수 |
| **`self_play.save_steps`** | 50,000 | Self-Play 체크포인트 저장 주기 |
| **`self_play.swap_steps`** | 2,000 | 상대 모델 교체 주기 |
| **`self_play.window`** | 10 | 유지하는 과거 체크포인트 수 |
| **`self_play.initial_elo`** | 1200.0 | 초기 ELO 점수 |

> ✏️ 하이퍼파라미터 참고: https://github.com/Unity-Technologies/ml-agents/blob/release_20_docs/docs/Training-Configuration-File.md  
> ⚠️ **관측/행동 공간 변경 금지**: AI vs AI 챌린지 규정 위반


In [8]:
import os

# YAML 설정 파일 내용
soccer_config = """\
behaviors:
  SoccerTwos:
    trainer_type: poca
    hyperparameters:
      batch_size: 2048
      buffer_size: 20480
      learning_rate: 0.0003
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: constant
    network_settings:
      normalize: false
      hidden_units: 512
      num_layers: 2
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.99
        strength: 1.0
    keep_checkpoints: 5
    max_steps: 5000000
    time_horizon: 1000
    summary_freq: 10000
    self_play:
      save_steps: 50000       # 50,000 스텝마다 체크포인트 저장
      team_change: 200000     # 200,000 스텝마다 팀 변경
      swap_steps: 2000        # 2,000 스텝마다 상대 모델 교체
      window: 10              # 최근 10개 체크포인트를 상대로 유지
      play_against_latest_model_ratio: 0.5  # 50% 확률로 최신 모델과 대전
      initial_elo: 1200.0    # 초기 ELO (2M 스텝 전까지는 ELO 하락 정상)
"""

config_dir  = '/content/ml-agents/config/poca'
config_path = f'{config_dir}/SoccerTwos.yaml'

os.makedirs(config_dir, exist_ok=True)
with open(config_path, 'w') as f:
    f.write(soccer_config)

print(f'✅ 설정 파일 생성: {config_path}')
print(soccer_config)


✅ 설정 파일 생성: /content/ml-agents/config/poca/SoccerTwos.yaml
behaviors:
  SoccerTwos:
    trainer_type: poca
    hyperparameters:
      batch_size: 2048
      buffer_size: 20480
      learning_rate: 0.0003
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: constant
    network_settings:
      normalize: false
      hidden_units: 512
      num_layers: 2
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.99
        strength: 1.0
    keep_checkpoints: 5
    max_steps: 5000000
    time_horizon: 1000
    summary_freq: 10000
    self_play:
      save_steps: 50000       # 50,000 스텝마다 체크포인트 저장
      team_change: 200000     # 200,000 스텝마다 팀 변경
      swap_steps: 2000        # 2,000 스텝마다 상대 모델 교체
      window: 10              # 최근 10개 체크포인트를 상대로 유지
      play_against_latest_model_ratio: 0.5  # 50% 확률로 최신 모델과 대전
      initial_elo: 1200.0    # 초기 ELO (2M 스텝 전까지는 ELO 하락 정상)



---
## 7. 에이전트 훈련

```
mlagents-learn <설정파일>      ← SoccerTwos.yaml
    --env=<환경실행파일>        ← SoccerTwos.x86_64 (Linux)
    --run-id=<실행ID>          ← 결과 폴더명
    --no-graphics              ← 렌더링 없이 실행 (Colab 필수)
    --resume                   ← 중단된 훈련 재개
```

> ⚠️ **ELO 주의사항**: 2M 스텝 이전에는 ELO가 1200 아래로 내려가도 정상입니다.  
> 에이전트가 처음에는 무작위로 움직이다가 점차 전략을 습득하기 때문입니다.

> ⚠️ **훈련 중단**: `Ctrl+C`를 한 번만 누르세요. ML-Agents가 최종 `.onnx` 파일을  
> 생성하고 종료할 시간이 필요합니다. 두 번 누르면 파일이 저장되지 않을 수 있습니다.

> 💡 **세션 재개**: Colab 세션이 끊겼을 경우 `--resume` 플래그를 추가하면  
> Drive에 저장된 체크포인트부터 이어서 훈련합니다.


In [ ]:
import subprocess, os

mlagents_learn = os.environ.get('MLAGENTS_LEARN',
    '/content/miniconda3/envs/mlagents/bin/mlagents-learn')
soccer_env     = os.environ.get('SOCCER_ENV',
    '/content/ml-agents/training-envs-executables/linux/SoccerTwos/SoccerTwos.x86_64')
config_path    = '/content/ml-agents/config/poca/SoccerTwos.yaml'
RUN_ID         = 'SoccerTwos'

# 실행 파일 존재 여부 확인
if not os.path.exists(soccer_env):
    # .x86_64 확장자 없이 시도
    alt_path = soccer_env.replace('.x86_64', '')
    if os.path.exists(alt_path):
        soccer_env = alt_path
    else:
        print(f'❌ 환경 실행 파일을 찾을 수 없습니다.')
        print(f'   확인 경로: {soccer_env}')
        print(f'   Cell 4 (환경 다운로드) 를 먼저 실행하세요.')
        raise FileNotFoundError(soccer_env)

print(f'훈련 시작: {RUN_ID}')
print(f'환경:     {soccer_env}')
print(f'설정:     {config_path}')
print(f'결과:     /content/ml-agents/results/{RUN_ID} → Drive에 자동 저장')
print('=' * 60)

# mlagents-learn 실행 (로그 실시간 출력)
# --resume: 중단된 훈련 재개 (첫 실행 시 오류가 나면 제거)
process = subprocess.Popen(
    [
        mlagents_learn, config_path,
        f'--env={soccer_env}',
        f'--run-id={RUN_ID}',
        '--no-graphics',
        '--resume'
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd='/content/ml-agents',  # ml-agents 디렉토리에서 실행
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()

if process.returncode == 0:
    print('\n✅ 훈련 완료!')
else:
    print(f'\n❌ 훈련 종료 (returncode={process.returncode})')
    print('Ctrl+C로 중단했거나 오류가 발생한 경우입니다.')


훈련 시작: SoccerTwos
환경:     /content/ml-agents/training-envs-executables/linux/SoccerTwos.x86_64
설정:     /content/ml-agents/config/poca/SoccerTwos.yaml
결과:     /content/ml-agents/results/SoccerTwos → Drive에 자동 저장

            ┐  ╖
        ╓╖╬│╡  ││╬╖╖
    ╓╖╬│││││┘  ╬│││││╬╖
 ╖╬│││││╬╜        ╙╬│││││╖╖                               ╗╗╗
 ╬╬╬╬╖││╦╖        ╖╬││╗╣╣╣╬      ╟╣╣╬    ╟╣╣╣             ╜╜╜  ╟╣╣
 ╬╬╬╬╬╬╬╬╖│╬╖╖╓╬╪│╓╣╣╣╣╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╒╣╣╖╗╣╣╣╗   ╣╣╣ ╣╣╣╣╣╣ ╟╣╣╖   ╣╣╣
 ╬╬╬╬┐  ╙╬╬╬╬│╓╣╣╣╝╜  ╫╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╟╣╣╣╙ ╙╣╣╣  ╣╣╣ ╙╟╣╣╜╙  ╫╣╣  ╟╣╣
 ╬╬╬╬┐     ╙╬╬╣╣      ╫╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╟╣╣╬   ╣╣╣  ╣╣╣  ╟╣╣     ╣╣╣┌╣╣╜
 ╬╬╬╜       ╬╬╣╣      ╙╝╣╣╬      ╙╣╣╣╗╖╓╗╣╣╣╜ ╟╣╣╬   ╣╣╣  ╣╣╣  ╟╣╣╦╓    ╣╣╣╣╣
 ╙   ╓╦╖    ╬╬╣╣   ╓╗╗╖            ╙╝╣╣╣╣╝╜   ╘╝╝╜   ╝╝╝  ╝╝╝   ╙╣╣╣    ╟╣╣╣
   ╩╬╬╬╬╬╬╦╦╬╬╣╣╗╣╣╣╣╣╣╣╝                                             ╫╣╣╣╣
      ╙╬╬╬╬╬╬╬╣╣╣╣╣╣╝╜
          ╙╬╬╬╣╣╣╜
             ╙
        
 Version information:
  ml-agents: 1.2.0.dev0,
  ml-agents-env

### 7-1. 훈련 결과 Drive에 백업

훈련 완료(또는 중단) 후 결과를 Drive에 백업합니다.  
Colab 세션이 끊기기 전에 실행하세요.


In [ ]:
import shutil, os

RUN_ID        = 'SoccerTwos'
LOCAL_RESULTS = os.environ.get('LOCAL_RESULTS', '/content/ml-agents/results')
DRIVE_RESULTS = os.environ.get('DRIVE_RESULTS',
    '/content/drive/MyDrive/RL_Course/Unit7_SoccerTwos/results')

src_dir = os.path.join(LOCAL_RESULTS, RUN_ID)
dst_dir = os.path.join(DRIVE_RESULTS, RUN_ID)

if os.path.exists(src_dir):
    print(f'Drive에 백업 중: {src_dir} → {dst_dir}')
    shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
    print('✅ 백업 완료!')
    print(f'   저장 위치: {dst_dir}')
else:
    print(f'❌ 훈련 결과 폴더가 없습니다: {src_dir}')
    print('   훈련 셀을 먼저 실행하세요.')

Drive에 백업 중: /content/ml-agents/results/SoccerTwos → /content/drive/MyDrive/RL_Course/Unit7_SoccerTwos/results/SoccerTwos
✅ 백업 완료!
   저장 위치: /content/drive/MyDrive/RL_Course/Unit7_SoccerTwos/results/SoccerTwos


---
## 8. 훈련 결과 확인 (TensorBoard)

**주요 지표:**
- `Environment/Cumulative Reward`: 누적 보상 (양수로 증가하면 좋음)
- `Self-play/ELO`: ELO 점수 (2M 스텝 전까지 하락은 정상)
- `Losses/Policy Loss`: 정책 손실
- `Losses/Value Loss`: 가치 함수 손실

> ELO 1200 이하로 떨어지는 것은 정상입니다. 훈련이 2M 스텝을 넘어가면 상승합니다.


In [15]:
import os

local_results = os.environ.get('LOCAL_RESULTS', '/content/ml-agents/results')

%load_ext tensorboard
%tensorboard --logdir {local_results}/SoccerTwos

<IPython.core.display.Javascript object>

---
## 9. Hugging Face Hub 업로드

`mlagents-push-to-hf`가 아래 작업을 자동으로 처리합니다:
- `SoccerTwos.onnx` 모델 파일 업로드
- 모델 카드 자동 생성 (`ML-Agents-SoccerTwos` 태그 포함)
- TensorBoard 로그 업로드

### AI vs AI 챌린지 참가 조건
업로드된 모델이 챌린지 풀에 자동 등록되려면:
1. 모델에 `ML-Agents-SoccerTwos` 태그가 있어야 함
2. `SoccerTwos.onnx` 파일이 존재해야 함

### 사전 준비
1. [HF 계정 생성](https://huggingface.co/join)
2. [Write 토큰 발급](https://huggingface.co/settings/tokens)


In [ ]:
import subprocess, os

conda_python = os.environ.get('CONDA_PYTHON',
    '/content/miniconda3/envs/mlagents/bin/python')

# ✏️ 본인의 HF 토큰 입력
HF_TOKEN = 'XXXXXXXXXX'

# conda 환경에서 HF 로그인
result = subprocess.run(
    [conda_python, '-c',
     f'from huggingface_hub import login; login(token="{HF_TOKEN}")'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('✅ HF 로그인 완료')
else:
    print('❌ HF 로그인 실패:', result.stderr[-300:])

# git credential 저장 (push-to-hf 내부에서 git을 사용)
!git config --global credential.helper store

✅ HF 로그인 완료


In [18]:
import os, shutil

# configuration.yaml 직접 생성
config_src  = '/content/ml-agents/config/poca/SoccerTwos.yaml'
config_dst  = '/content/ml-agents/results/SoccerTwos/configuration.yaml'

shutil.copy(config_src, config_dst)
print(f'✅ configuration.yaml 생성: {config_dst}')

# results 폴더 내용 확인
for f in os.listdir('/content/ml-agents/results/SoccerTwos'):
    print(f'  {f}')

✅ configuration.yaml 생성: /content/ml-agents/results/SoccerTwos/configuration.yaml
  SoccerTwos
  configuration.yaml
  run_logs


In [19]:
import subprocess, os

mlagents_push = os.environ.get('MLAGENTS_PUSH',
    '/content/miniconda3/envs/mlagents/bin/mlagents-push-to-hf')

# ✏️ 아래 값을 본인 정보로 수정하세요.
RUN_ID       = 'SoccerTwos'
LOCAL_DIR    = os.path.join(
    os.environ.get('LOCAL_RESULTS', '/content/ml-agents/results'), RUN_ID
)
HF_USERNAME  = 'DitDahDitDit'  # ← HF 사용자명 변경
REPO_ID      = f'{HF_USERNAME}/poca-{RUN_ID}'
COMMIT_MSG   = 'First Push'

print(f'업로드: {LOCAL_DIR} → {REPO_ID}')

process = subprocess.Popen(
    [
        mlagents_push,
        f'--run-id={RUN_ID}',
        f'--local-dir={LOCAL_DIR}',
        f'--repo-id={REPO_ID}',
        f'--commit-message={COMMIT_MSG}',
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    cwd='/content/ml-agents',
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()

if process.returncode == 0:
    print(f'\n✅ 업로드 완료!')
    print(f'   모델 확인: https://huggingface.co/{REPO_ID}')
else:
    print(f'\n❌ 업로드 실패 (returncode={process.returncode})')

업로드: /content/ml-agents/results/SoccerTwos → DitDahDitDit/poca-SoccerTwos


[INFO] This function will create a model card and upload your SoccerTwos into HuggingFace Hub. This is a work in progress: If you encounter a bug, please send open an issue
[INFO] Pushing repo SoccerTwos to the Hugging Face Hub
[INFO] Your model is pushed to the hub. You can view your model here: https://huggingface.co/DitDahDitDit/poca-SoccerTwos

✅ 업로드 완료!
   모델 확인: https://huggingface.co/DitDahDitDit/poca-SoccerTwos


---
## 10. AI vs AI 챌린지 참가 확인

업로드 후 모델이 챌린지에 정상 등록됐는지 확인합니다.

### 체크리스트

**1. 모델 태그 확인**  
HF Hub 모델 페이지에서 `ML-Agents-SoccerTwos` 태그가 있는지 확인하세요.  
없으면 README.md에 직접 추가하면 됩니다.

**2. SoccerTwos.onnx 파일 확인**  
모델 파일 목록에 `SoccerTwos.onnx`가 있어야 합니다.

**3. 리더보드 등록 확인** (최대 4시간 소요)  
4시간마다 매치메이킹이 실행되어 자동으로 리더보드에 등록됩니다.

### 유용한 링크
- 리더보드: https://huggingface.co/spaces/huggingface-projects/AIvsAI-SoccerTwos
- 시각화 데모: https://huggingface.co/spaces/unity/ML-Agents-SoccerTwos
- 매치 분석: https://huggingface.co/spaces/cyllum/soccertwos-analytics
- 베이스라인 모델: https://huggingface.co/unity/MLAgents-SoccerTwos


In [20]:
# 업로드된 모델 및 챌린지 링크 출력
HF_USERNAME = 'DitDahDitDit'  # ✏️ 본인 username
RUN_ID      = 'SoccerTwos'
REPO_ID     = f'{HF_USERNAME}/poca-{RUN_ID}'

print('=' * 60)
print('📋 확인 링크')
print('=' * 60)
print(f'내 모델:    https://huggingface.co/{REPO_ID}')
print(f'리더보드:   https://huggingface.co/spaces/huggingface-projects/AIvsAI-SoccerTwos')
print(f'시각화:     https://huggingface.co/spaces/unity/ML-Agents-SoccerTwos')
print(f'매치 분석:  https://huggingface.co/spaces/cyllum/soccertwos-analytics')
print()
print('⚠️  리더보드 반영까지 최대 4시간이 걸릴 수 있습니다.')
print('⚠️  2M 스텝 이전에는 ELO가 1200 아래로 내려가는 것이 정상입니다.')

📋 확인 링크
내 모델:    https://huggingface.co/DitDahDitDit/poca-SoccerTwos
리더보드:   https://huggingface.co/spaces/huggingface-projects/AIvsAI-SoccerTwos
시각화:     https://huggingface.co/spaces/unity/ML-Agents-SoccerTwos
매치 분석:  https://huggingface.co/spaces/cyllum/soccertwos-analytics

⚠️  리더보드 반영까지 최대 4시간이 걸릴 수 있습니다.
⚠️  2M 스텝 이전에는 ELO가 1200 아래로 내려가는 것이 정상입니다.
